# Worksheet: Classification I: training & predicting

This worksheet covers the [Classification I: training & predicting](https://python.datasciencebook.ca/classification1.html) chapter of the online textbook, which also lists the learning objectives for this worksheet. You should read the textbook chapter before attempting this worksheet. 

In [132]:
### Run this cell before continuing
import random

import altair as alt
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics.pairwise import euclidean_distances
from sklearn import set_config

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

**Question 0.1** Multiple Choice: 
<br> {points: 1}

**Which of the following statements is NOT true of a training data set (in the context of classification)?**

A. A training data set is a collection of observations for which we know the true classes.

B. We can use a training set to explore and build our classifier.

C. The training data set is the underlying collection of observations for which we don't know the true classes.

*Assign your answer to an object called `answer0_1`. Make sure the correct answer is an uppercase letter. Remember to surround your answer with quotation marks (e.g. "D").*

In [133]:
answer0_1 = 'C'

In [134]:
from hashlib import sha1
assert sha1(str(type(answer0_1)).encode("utf-8")+b"59e8").hexdigest() == "f70a6af77babf11afa3a50f7a15e532aa7baf3f3", "type of answer0_1 is not str. answer0_1 should be an str"
assert sha1(str(len(answer0_1)).encode("utf-8")+b"59e8").hexdigest() == "3b3b8f9d183db109adfa13225d741063b08dff85", "length of answer0_1 is not correct"
assert sha1(str(answer0_1.lower()).encode("utf-8")+b"59e8").hexdigest() == "6edfdac10655fefa98b28f34c9d1f860dcce8ff7", "value of answer0_1 is not correct"
assert sha1(str(answer0_1).encode("utf-8")+b"59e8").hexdigest() == "9dd39830e122b8184b1799ac7d2e687f95a317b7", "correct string value of answer0_1 but incorrect case of letters"

print('Success!')

Success!


**Question 0.2** Multiple Choice
<br> {points: 1}

(Adapted from James et al, "[An introduction to statistical learning](http://www-bcf.usc.edu/~gareth/ISL/)" (page 53))

Consider the scenario below: 

We collect data on 20 similar products. For each product we have recorded whether it was a success or failure (labelled as such by the Sales team), price charged for the product, marketing budget, competition price, customer data, and ten other variables. 

**Which of the following is a classification problem?**

A. We are interested in comparing the profit margins for products that are a success and products that are a failure. 

B. We are considering launching a new product and wish to know whether it will be a success or a failure. 

C. We wish to group customers based on their preferences and use that knowledge to develop targeted marketing programs. 

*Assign your answer to an object called `answer0_2`. Make sure the correct answer is an uppercase letter. Remember to surround your answer with quotation marks (e.g. "F").*

In [135]:
answer0_2 = 'B'

In [136]:
from hashlib import sha1
assert sha1(str(type(answer0_2)).encode("utf-8")+b"b9cdb").hexdigest() == "351b6021c8f87571484eee4e4c7bf0a0b7ed21fc", "type of answer0_2 is not str. answer0_2 should be an str"
assert sha1(str(len(answer0_2)).encode("utf-8")+b"b9cdb").hexdigest() == "439859842b2ff5343f142aebc166bbd2f5c1ba63", "length of answer0_2 is not correct"
assert sha1(str(answer0_2.lower()).encode("utf-8")+b"b9cdb").hexdigest() == "eac7d86ae102d0cf10bcd9c8da29efa7bd262db9", "value of answer0_2 is not correct"
assert sha1(str(answer0_2).encode("utf-8")+b"b9cdb").hexdigest() == "219cf8d4b385560346b8cfeb877dfdfc50ca9e69", "correct string value of answer0_2 but incorrect case of letters"

print('Success!')

Success!


## 1. Breast Cancer Data Set 
We will work with the breast cancer data from this week's pre-reading. 
> Note that the breast cancer data in this worksheet have been **standardized (centred and scaled)** for you already. We will implement these steps in future worksheets/tutorials later, but for now, know the data has been standardized. Therefore the variables are unitless and hence why we have zero and negative values for variables like Radius. 

**Question 1.0**
<br> {points: 1}

Read the `clean-wdbc-data.csv` file (found in the `data` directory) using the `pd.read_csv` function into the notebook and store it as a data frame. *Name it `cancer`.*

In [137]:
cancer = pd.read_csv('data/clean-wdbc-data.csv')
cancer

,ID,Class,Radius,Texture,Perimeter,Area,Smoothness,Compactness,Concavity,Concave_points,Symmetry,Fractal_dimension
0,842302,M,1.885031,-1.358098,2.301575,1.999478,1.306537,2.614365,2.107672,2.294058,2.748204,1.935312
1,842517,M,1.804340,-0.368879,1.533776,1.888827,-0.375282,-0.430066,-0.146620,1.086129,-0.243675,0.280943
2,84300903,M,1.510541,-0.023953,1.346291,1.455004,0.526944,1.081980,0.854222,1.953282,1.151242,0.201214
3,84348301,M,-0.281217,0.133866,-0.249720,-0.549538,3.391291,3.889975,1.987839,2.173873,6.040726,4.930672
4,84358402,M,1.297434,-1.465481,1.337363,1.219651,0.220362,-0.313119,0.612640,0.728618,-0.867590,-0.396751
...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,1.899514,0.117596,1.751022,2.013529,0.378033,-0.273077,0.663928,1.627719,-1.358963,-0.708467
565,926682,M,1.535369,2.045599,1.420690,1.493644,-0.690623,-0.394473,0.236365,0.733182,-0.531387,-0.973122
566,926954,M,0.560868,1.373645,0.578492,0.427529,-0.808876,0.350427,0.326479,0.413705,-1.103578,-0.318129
567,927241,M,1.959515,2.235958,2.301575,1.651717,1.429169,3.901415,3.194794,2.287972,1.917396,2.217684


In [138]:
from hashlib import sha1
assert sha1(str(type(cancer is None)).encode("utf-8")+b"1edfa").hexdigest() == "f89107c5738f5567ac4ce7e619af326d8dc7e7e4", "type of cancer is None is not bool. cancer is None should be a bool"
assert sha1(str(cancer is None).encode("utf-8")+b"1edfa").hexdigest() == "71bbe216c3f112174b74da6122dd837cb4abaafa", "boolean value of cancer is None is not correct"

assert sha1(str(type(cancer)).encode("utf-8")+b"1edfb").hexdigest() == "5f4717efa0f9568127506afdab187398929a3f76", "type of type(cancer) is not correct"

assert sha1(str(type(cancer.shape)).encode("utf-8")+b"1edfc").hexdigest() == "6790c026fc62f7f025cc5dfb65b5589e70c94f24", "type of cancer.shape is not tuple. cancer.shape should be a tuple"
assert sha1(str(len(cancer.shape)).encode("utf-8")+b"1edfc").hexdigest() == "e400bacf4406589a930f6cb04146f55ae09adbe1", "length of cancer.shape is not correct"
assert sha1(str(sorted(map(str, cancer.shape))).encode("utf-8")+b"1edfc").hexdigest() == "bc58412175f96aad1b5f00e35cbf014630e9667a", "values of cancer.shape are not correct"
assert sha1(str(cancer.shape).encode("utf-8")+b"1edfc").hexdigest() == "03f45e6b2934a9d89a82761d8cd384e41a96882e", "order of elements of cancer.shape is not correct"

assert sha1(str(type(sum(cancer.Area))).encode("utf-8")+b"1edfd").hexdigest() == "8459034dfcce3dc398256d3235e7ea1c0a6eab66", "type of sum(cancer.Area) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(cancer.Area), 2)).encode("utf-8")+b"1edfd").hexdigest() == "578eeb3cce10656ea3740570a63c462e74b5bee3", "value of sum(cancer.Area) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(cancer.columns.values)).encode("utf-8")+b"1edfe").hexdigest() == "ec1b028616e5bf668b318569d326edbe4fa5792d", "type of cancer.columns.values is not correct"
assert sha1(str(cancer.columns.values).encode("utf-8")+b"1edfe").hexdigest() == "46e462ccc418670863709af1c7c6c89e4aa501be", "value of cancer.columns.values is not correct"

assert sha1(str(type(cancer['Class'].dtype)).encode("utf-8")+b"1edff").hexdigest() == "0c82f629a8e37cf9fc16959e34213e0fa89a6a68", "type of cancer['Class'].dtype is not correct"
assert sha1(str(cancer['Class'].dtype).encode("utf-8")+b"1edff").hexdigest() == "852a6171a68036a67ec486822760a69c8cd61e0d", "value of cancer['Class'].dtype is not correct"

print('Success!')

Success!


**Question 1.1** True or False: 
<br> {points: 1}

After looking at the first six rows of the `cancer` data fame, suppose we asked you to predict the variable "area" for a new observation. **Is this a classification problem?**

*Assign your answer to an object called `answer1_1`. Make sure the correct answer is a boolean. i.e. `True` or `False`.*

In [139]:
answer1_1 = False
# bc not much of trend seen so far *check for correctness

In [140]:
from hashlib import sha1
assert sha1(str(type(answer1_1)).encode("utf-8")+b"3524b").hexdigest() == "96e967f4261014f1a2a76ba5f230d7b7e85abe83", "type of answer1_1 is not bool. answer1_1 should be a bool"
assert sha1(str(answer1_1).encode("utf-8")+b"3524b").hexdigest() == "88e4fbb75d430084961a28b324179d14f4999b12", "boolean value of answer1_1 is not correct"

print('Success!')

Success!


**Question 1.2** 
<br> {points: 1}

Create a scatterplot of the data with `Symmetry` on the x-axis and `Radius` on the y-axis. Modify your aesthetics by colouring for `Class`. As you create this plot, ensure you follow the guidelines for creating effective visualizations. In particular, note in the chart axis titles whether the data is standardized or not. Add an opacity level of 0.5 to the graphical mark. You should also replace the values in the dataframe's `Class` column from `'M'` to `'Malignant'` and from `'B'` to `'Benign'`. 

*Assign your plot to an object called `cancer_plot`.*

In [141]:
cancer["Class"] = cancer["Class"].replace({
    'M': 'Malignant',
    'B': 'Benign'})

cancer_plot = alt.Chart(cancer).mark_point(opacity=0.5).encode(
    x=alt.X('Symmetry').title('Symmetry (standardized)'),
    y=alt.Y('Radius').title('Radius (standardized)'),
    color = alt.Color('Class').title('Diagnosis'),
)

cancer_plot

alt.Chart(...)

In [142]:
from hashlib import sha1
assert sha1(str(type(cancer['Class'].unique())).encode("utf-8")+b"7038e").hexdigest() == "2c870d7a3657b742557d66961de4a4891ee76aa2", "type of cancer['Class'].unique() is not correct"
assert sha1(str(cancer['Class'].unique()).encode("utf-8")+b"7038e").hexdigest() == "7be82206c225ab0f0a4ffad7c1488a64143481f7", "value of cancer['Class'].unique() is not correct"

assert sha1(str(type(cancer_plot is None)).encode("utf-8")+b"7038f").hexdigest() == "5f05f0c0e171e12d0d2b2a22d2789f0d9a8e342e", "type of cancer_plot is None is not bool. cancer_plot is None should be a bool"
assert sha1(str(cancer_plot is None).encode("utf-8")+b"7038f").hexdigest() == "515f1ad42fa781dff7c9975c8ded7f317ce95332", "boolean value of cancer_plot is None is not correct"

assert sha1(str(type(cancer_plot.encoding.x['shorthand'])).encode("utf-8")+b"70390").hexdigest() == "1b054e65bf0aac12e848da1d2afa2b12a5226b61", "type of cancer_plot.encoding.x['shorthand'] is not str. cancer_plot.encoding.x['shorthand'] should be an str"
assert sha1(str(len(cancer_plot.encoding.x['shorthand'])).encode("utf-8")+b"70390").hexdigest() == "dd930b7fa9aa4d16b4d1a89c7a39089300827489", "length of cancer_plot.encoding.x['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.x['shorthand'].lower()).encode("utf-8")+b"70390").hexdigest() == "dd17739f27d755296153ed0ec742286fd3b39cb2", "value of cancer_plot.encoding.x['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.x['shorthand']).encode("utf-8")+b"70390").hexdigest() == "d4212b1d5679f4601309609b6ea38ffb2d4ed07c", "correct string value of cancer_plot.encoding.x['shorthand'] but incorrect case of letters"

assert sha1(str(type(cancer_plot.encoding.y['shorthand'])).encode("utf-8")+b"70391").hexdigest() == "6fac44d279065f2a3f185c5a884fd9df6dd81cfb", "type of cancer_plot.encoding.y['shorthand'] is not str. cancer_plot.encoding.y['shorthand'] should be an str"
assert sha1(str(len(cancer_plot.encoding.y['shorthand'])).encode("utf-8")+b"70391").hexdigest() == "df067cfef2a7c5291dd0f9acc53fe92d50ccf1fe", "length of cancer_plot.encoding.y['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.y['shorthand'].lower()).encode("utf-8")+b"70391").hexdigest() == "edf8bc4ede29cf991a36bb6d8b38713806753b47", "value of cancer_plot.encoding.y['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.y['shorthand']).encode("utf-8")+b"70391").hexdigest() == "e3f5fd9a0d9f3c695791e536282f29ee7d2c6b4a", "correct string value of cancer_plot.encoding.y['shorthand'] but incorrect case of letters"

assert sha1(str(type(cancer_plot.encoding.color['shorthand'])).encode("utf-8")+b"70392").hexdigest() == "a1514e0d638e7fa47039d17e310bba44e6a8ff2c", "type of cancer_plot.encoding.color['shorthand'] is not str. cancer_plot.encoding.color['shorthand'] should be an str"
assert sha1(str(len(cancer_plot.encoding.color['shorthand'])).encode("utf-8")+b"70392").hexdigest() == "a43d370b386a7d21c0ac8e73bdc942b85ee93cae", "length of cancer_plot.encoding.color['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.color['shorthand'].lower()).encode("utf-8")+b"70392").hexdigest() == "a2e54ddb2a451b8647df9f5b9aae970b19620f66", "value of cancer_plot.encoding.color['shorthand'] is not correct"
assert sha1(str(cancer_plot.encoding.color['shorthand']).encode("utf-8")+b"70392").hexdigest() == "dfff632fd3104b75a651bf07567aa5d572835f70", "correct string value of cancer_plot.encoding.color['shorthand'] but incorrect case of letters"

assert sha1(str(type(cancer_plot.mark)).encode("utf-8")+b"70393").hexdigest() == "e736aec18d8c2224b69bfe505db7d3968a4a2c7e", "type of cancer_plot.mark is not correct"
assert sha1(str(cancer_plot.mark).encode("utf-8")+b"70393").hexdigest() == "1746955c8bc53ef47b4cf0267c98ce50ce3b184a", "value of cancer_plot.mark is not correct"

assert sha1(str(type(isinstance(cancer_plot.encoding.color['title'], str))).encode("utf-8")+b"70394").hexdigest() == "40775fb7d185dcb2ad7acf54925bb5075bd50aca", "type of isinstance(cancer_plot.encoding.color['title'], str) is not bool. isinstance(cancer_plot.encoding.color['title'], str) should be a bool"
assert sha1(str(isinstance(cancer_plot.encoding.color['title'], str)).encode("utf-8")+b"70394").hexdigest() == "e42ecc8f1d08c9a9cbb062b31d6180e6271e740b", "boolean value of isinstance(cancer_plot.encoding.color['title'], str) is not correct"

assert sha1(str(type(isinstance(cancer_plot.encoding.x['title'], str))).encode("utf-8")+b"70395").hexdigest() == "711e00550c10a33b1b90544076b3b8d4a89d23bf", "type of isinstance(cancer_plot.encoding.x['title'], str) is not bool. isinstance(cancer_plot.encoding.x['title'], str) should be a bool"
assert sha1(str(isinstance(cancer_plot.encoding.x['title'], str)).encode("utf-8")+b"70395").hexdigest() == "b697a6b22f874c7f85b3e081a24d1b5cdd084659", "boolean value of isinstance(cancer_plot.encoding.x['title'], str) is not correct"

assert sha1(str(type(isinstance(cancer_plot.encoding.y['title'], str))).encode("utf-8")+b"70396").hexdigest() == "37f686ca3136e2e102714b66fd51cf3ef4703fc6", "type of isinstance(cancer_plot.encoding.y['title'], str) is not bool. isinstance(cancer_plot.encoding.y['title'], str) should be a bool"
assert sha1(str(isinstance(cancer_plot.encoding.y['title'], str)).encode("utf-8")+b"70396").hexdigest() == "d0eb5c4439424db54df14662a17296d7dc74ff3d", "boolean value of isinstance(cancer_plot.encoding.y['title'], str) is not correct"

print('Success!')

Success!


**Question 1.3** 
<br> {points: 1}

Just by looking at the scatterplot above, how would you classify an observation with `Symmetry` = 1 and `Radius` = 1 (Benign or Malignant)?

*Assign your answer to an object called `answer1_3`. Make sure the correct answer is written fully. Remember to surround your answer with quotation marks (e.g. "Benign" / "Malignant").*

In [143]:
answer1_3 = "Malignant"

In [144]:
from hashlib import sha1
assert sha1(str(type(answer1_3)).encode("utf-8")+b"192dd").hexdigest() == "c22b7e131d7091f92de118d328cf99ebfc373e56", "type of answer1_3 is not str. answer1_3 should be an str"
assert sha1(str(len(answer1_3)).encode("utf-8")+b"192dd").hexdigest() == "426e383fda7dfde77b6cbef2a70be0047a593331", "length of answer1_3 is not correct"
assert sha1(str(answer1_3.lower()).encode("utf-8")+b"192dd").hexdigest() == "04f0ffa0d0870e9e4d9a8bf7bb1ed255f9165a52", "value of answer1_3 is not correct"
assert sha1(str(answer1_3).encode("utf-8")+b"192dd").hexdigest() == "11394ec2da0fa66e3dd679f582c3d6024557dc1a", "correct string value of answer1_3 but incorrect case of letters"

print('Success!')

Success!


We will now compute the distance between the first and second observation in the breast cancer dataset using the explanatory variables/predictors `Symmetry` and `Radius`. Recall we can calculate the distance between two points using the following formula:

$$Distance = \sqrt{(a_x -b_x)^2 + (a_y - b_y)^2}$$

**Question 1.4** 
<br> {points: 1}

First, extract the coordinates for the two observations and assign them to objects called: 

- `ax` (Symmetry value for the first row)
- `ay` (Radius value for the first row)
- `bx` (Symmetry value for the second row)
- `by` (Radius value for the second row).

*Scaffolding for `ax` is given*.

In [145]:
ax = cancer.loc[0, "Symmetry"] # selecting the first observation from cancer and pulling the value from the Symmetry column as a numeric value only
ay = cancer.loc[0, "Radius"]
bx = cancer.loc[1, "Symmetry"] 
by = cancer.loc[1, "Radius"]

In [146]:
from hashlib import sha1
assert sha1(str(type(ax)).encode("utf-8")+b"32651").hexdigest() == "1f22f15027a2de0bde9fb0d308e055facfbf3048", "type of ax is not correct"
assert sha1(str(ax).encode("utf-8")+b"32651").hexdigest() == "5e66f41a61031f51d3ca5bde161a049f73fa0832", "value of ax is not correct"

assert sha1(str(type(ay)).encode("utf-8")+b"32652").hexdigest() == "7d58a16b2f107511fcc0771f2dc9a98df92f9b84", "type of ay is not correct"
assert sha1(str(ay).encode("utf-8")+b"32652").hexdigest() == "099e4ade255d067438bd67328df19295f3d8a168", "value of ay is not correct"

assert sha1(str(type(bx)).encode("utf-8")+b"32653").hexdigest() == "f14665564fd3232c8dccb01df0948cedc3e1c0f3", "type of bx is not correct"
assert sha1(str(bx).encode("utf-8")+b"32653").hexdigest() == "1b41f3e916d32900d08be174627a8d4acd86af3d", "value of bx is not correct"

assert sha1(str(type(by)).encode("utf-8")+b"32654").hexdigest() == "29d2aff41f6f3cdc36a9130065e1f2c3da808166", "type of by is not correct"
assert sha1(str(by).encode("utf-8")+b"32654").hexdigest() == "cff9ad44834cf4ef678d982ebb2749462652e5f0", "value of by is not correct"

print('Success!')

Success!


**Question 1.5**
<br> {points: 1}

Plug the coordinates into the distance equation. Hint: `**` is the exponent symbol in Python. For example, 2**4 computes 2 raised to the power of 4 in Python.

*Assign your answer to an object called `answer1_5`.*

Fill in the `___` in the cell below. 

In [147]:
answer1_5 = ((ax - bx)**2 + (ay - by)**2)**0.5


answer1_5

np.float64(2.9929672987960956)

In [148]:
from hashlib import sha1
assert sha1(str(type(answer1_5)).encode("utf-8")+b"81cea").hexdigest() == "80d7f3967b5b43f46f45380d15edafeaadd9aa6d", "type of answer1_5 is not correct"
assert sha1(str(answer1_5).encode("utf-8")+b"81cea").hexdigest() == "6fbac4f550b8d96c40f78c65a3078a2eaa755667", "value of answer1_5 is not correct"

print('Success!')

Success!


**Question 1.6**
<br> {points: 1}

Now we'll do the same thing *with 3 explanatory variables/predictors*: `Symmetry`, `Radius` and `Concavity`. Again, use the first two rows in the data set as the points you are calculating the distance between (point $a$ is row 0, and point $b$ is row 1).


*Find the coordinates for the third variable (Concavity) and assign them to objects called `az` and `bz`. Use the scaffolding given in **Question 1.4** as a guide.*

In [149]:
az = cancer.loc[0, "Concavity"] # selecting the first observation from cancer and pulling the value from the Symmetry column as a numeric value only
bz = cancer.loc[1, "Concavity"]

In [150]:
from hashlib import sha1
assert sha1(str(type(az)).encode("utf-8")+b"cef26").hexdigest() == "3307950086aeb61ac76111a6dd7fd52812e0707b", "type of az is not correct"
assert sha1(str(az).encode("utf-8")+b"cef26").hexdigest() == "3e8e5bed3e7c1c569d0cb207d27ecfced044b57c", "value of az is not correct"

assert sha1(str(type(bz)).encode("utf-8")+b"cef27").hexdigest() == "ee9869bc1528396d06e11b686e5fea34ea32c643", "type of bz is not correct"
assert sha1(str(bz).encode("utf-8")+b"cef27").hexdigest() == "1ab9403a22cae8a23dea0228f2f4aeb87949b030", "value of bz is not correct"

print('Success!')

Success!


**Question 1.7**
<br> {points: 1}

Again, calculate the distance between the first and second observation in the breast cancer dataset using 3 explanatory variables/predictors: `Symmetry`, `Radius` and `Concavity`.

*Assign your answer to an object called `answer1_7`. Use the scaffolding given to calculate `answer1_5` as a guide.*

In [151]:
answer1_7 = ((az - bz)**2 + (ax - bx)**2 + (ay - by)**2)**0.5
answer1_7

np.float64(3.7469567200527925)

In [152]:
from hashlib import sha1
assert sha1(str(type(round(answer1_7, 2))).encode("utf-8")+b"f198c").hexdigest() == "eff15a08c58efd6c23676c72d27648bcf46da228", "type of round(answer1_7, 2) is not correct"
assert sha1(str(round(answer1_7, 2)).encode("utf-8")+b"f198c").hexdigest() == "33c74f0f23a16e9972bb7b9cf81daf4137d91264", "value of round(answer1_7, 2) is not correct"

print('Success!')

Success!


**Question 1.8**
<br> {points: 1}

Let's do this without explicitly making coordinate variables!

Create a pandas series of the coordinates for each point. Name one series `point_a` and the other series `point_b`. Within the series, the order of coordinates should be: `Symmetry`, `Radius`, `Concavity`.

Fill in the `___` in the cell below.


In [153]:
# This is only the scaffolding for one point (you need to make another one for row number 1)

point_a =  cancer.loc[0, ["Symmetry", "Radius", "Concavity"]]
point_b =  cancer.loc[1, ["Symmetry", "Radius", "Concavity"]]



print(point_a)
print(point_b)

Symmetry     2.748204
Radius       1.885031
Concavity    2.107672
Name: 0, dtype: object
Symmetry    -0.243675
Radius        1.80434
Concavity    -0.14662
Name: 1, dtype: object


In [154]:
from hashlib import sha1
assert sha1(str(type(round(sum(point_a), 2))).encode("utf-8")+b"f3e40").hexdigest() == "7469d896db4c21f0066c96e60188fd82f3e26789", "type of round(sum(point_a), 2) is not correct"
assert sha1(str(round(sum(point_a), 2)).encode("utf-8")+b"f3e40").hexdigest() == "4fe65438541d875a78f83b05a0d8aad9cdc03542", "value of round(sum(point_a), 2) is not correct"

assert sha1(str(type(round(sum(point_b), 2))).encode("utf-8")+b"f3e41").hexdigest() == "dfd27081f3392d2d05ab01943b9ac21653d84500", "type of round(sum(point_b), 2) is not correct"
assert sha1(str(round(sum(point_b), 2)).encode("utf-8")+b"f3e41").hexdigest() == "a5296f79d982da1d374ee8a9ff0d8f482ed60d98", "value of round(sum(point_b), 2) is not correct"

print('Success!')

Success!


**Question 1.9**
<br> {points: 1}

Compute the squared differences between the two series, `point_a` and `point_b`. The result should be a series of length 3 named `dif_square`. *Hint: `**` is the exponent symbol in Python.*

In [155]:
symmetry_diff = (point_b["Symmetry"] - point_a["Symmetry"])**2 
radius_diff = (point_b["Radius"] - point_a["Radius"])**2 
concaivty_diff = (point_b["Concavity"] - point_a["Concavity"])**2 

In [156]:
dif_square = [symmetry_diff, radius_diff, concaivty_diff]

In [157]:
from hashlib import sha1
assert sha1(str(type(sum(dif_square))).encode("utf-8")+b"70681").hexdigest() == "f70a2394d44f5f8a4be80f78641b4fb996ae4247", "type of sum(dif_square) is not correct"
assert sha1(str(sum(dif_square)).encode("utf-8")+b"70681").hexdigest() == "f149d38ba9e6f7578aa558f78c9c3517959ddff4", "value of sum(dif_square) is not correct"

print('Success!')

Success!


**Question 1.09.1**
<br> {points: 1}

Sum the squared differences between the two points, `point_a` and `point_b`. The result should be a single number named `dif_sum`. 

*Hint: the `sum` dataframe method in Python returns the sum of the elements in a series*

In [158]:
dif_sum = sum(dif_square)
dif_sum

np.float64(14.039684661948781)

In [159]:
from hashlib import sha1
assert sha1(str(type(dif_sum)).encode("utf-8")+b"a3473").hexdigest() == "4538830cea59c7779bcb2aa16a18a424684bdbef", "type of dif_sum is not correct"
assert sha1(str(dif_sum).encode("utf-8")+b"a3473").hexdigest() == "07cd77f6673541a859280c2d89ab5d2ccad9d6e3", "value of dif_sum is not correct"

print('Success!')

Success!


**Question 1.09.2**
<br> {points: 1}

Square root the sum of your squared differences. The result should be a number named `root_dif_sum`. 

In [160]:
root_dif_sum = (dif_sum)**0.5
root_dif_sum

np.float64(3.7469567200527925)

In [161]:
from hashlib import sha1
assert sha1(str(type(root_dif_sum)).encode("utf-8")+b"3110b").hexdigest() == "ee6eda00442436e0cc3a290b8c1299288e16874d", "type of root_dif_sum is not correct"
assert sha1(str(root_dif_sum).encode("utf-8")+b"3110b").hexdigest() == "fc11816984dba681433728c14ae0ea28339c5d7d", "value of root_dif_sum is not correct"

print('Success!')

Success!


**Question 1.09.3**
<br> {points: 1}

If we have more than a few points, calculating distances as we did in the previous questions is VERY tedious. Let's use the `euclidean_distances` function from `scikit-learn` package to find the distance between the first and second observation in the breast cancer dataset using Symmetry, Radius and Concavity. 

Fill in the `___` in the cell below. 

*Assign your answer to an object called `dist_cancer_two_rows`. Note that the euclidean_distances function will return four values, one for each pair of combinations of the coordinates for the two points; we will see another example of this and describe it more in detail later in this worksheet.*

In [162]:
dist_cancer_two_rows = euclidean_distances(
    cancer.loc[0:1, ["Symmetry", "Radius", "Concavity"]]
)

dist_cancer_two_rows

array([[0.        , 3.74695672],
       [3.74695672, 0.        ]])

In [163]:
from hashlib import sha1
assert sha1(str(type(dist_cancer_two_rows[0, 0])).encode("utf-8")+b"1e910").hexdigest() == "731f85b1cde09b637c6a573f595f011e857e7620", "type of dist_cancer_two_rows[0, 0] is not correct"
assert sha1(str(dist_cancer_two_rows[0, 0]).encode("utf-8")+b"1e910").hexdigest() == "97b0e8e12886ffa0efcbe077a6c8c799812d06a5", "value of dist_cancer_two_rows[0, 0] is not correct"

assert sha1(str(type(dist_cancer_two_rows[0, 1])).encode("utf-8")+b"1e911").hexdigest() == "b730871489cd444c1fe2ce3854dd65c05209a18d", "type of dist_cancer_two_rows[0, 1] is not correct"
assert sha1(str(dist_cancer_two_rows[0, 1]).encode("utf-8")+b"1e911").hexdigest() == "fe560ad46e82f22740932923da9e324e42f388d9", "value of dist_cancer_two_rows[0, 1] is not correct"

assert sha1(str(type(dist_cancer_two_rows[1, 0])).encode("utf-8")+b"1e912").hexdigest() == "05a27c3f2dba94eb3401af56f1bb7db6ede17197", "type of dist_cancer_two_rows[1, 0] is not correct"
assert sha1(str(dist_cancer_two_rows[1, 0]).encode("utf-8")+b"1e912").hexdigest() == "bb7d1acf3f2770300dec0cec5468820293481d53", "value of dist_cancer_two_rows[1, 0] is not correct"

assert sha1(str(type(dist_cancer_two_rows[1, 1])).encode("utf-8")+b"1e913").hexdigest() == "69063759774dc948637592f06b150bea4c3cfba2", "type of dist_cancer_two_rows[1, 1] is not correct"
assert sha1(str(dist_cancer_two_rows[1, 1]).encode("utf-8")+b"1e913").hexdigest() == "87496a985f0bdcce066a1c6bf3364df6f5a90a61", "value of dist_cancer_two_rows[1, 1] is not correct"

print('Success!')

Success!


**Question 1.09.4** True or False: 
<br> {points: 1}

Compare `answer1_7`, `root_dif_sum`, and `dist_cancer_two_rows`. 

**Are they all the same value?** 

*Assign your answer to an object called `answer1_09_4`. Make sure the correct answer is a boolean. i.e. `True` or `False`.*

In [164]:
answer1_09_4 = True 

In [165]:
from hashlib import sha1
assert sha1(str(type(answer1_09_4)).encode("utf-8")+b"8637").hexdigest() == "0a5bb53e49014286acc41d42e260622200353667", "type of answer1_09_4 is not bool. answer1_09_4 should be a bool"
assert sha1(str(answer1_09_4).encode("utf-8")+b"8637").hexdigest() == "62b0f58a4aa4831a4eb476b4202701b1776bbec8", "boolean value of answer1_09_4 is not correct"

print('Success!')

Success!


## 2. Classification - A Simple Example Done Manually

**Question 2.0.0**
<br> {points: 1}

Let's take a random sample of 5 observations from the breast cancer dataset using the `sample` dataframe method. To make this random sample reproducible, we will use `random_state=20` inside the `sample` function. This means that the random number generator will start at the same point each time when we run the code and we will always get back the same random samples. 

We will focus on the predictors Symmetry and Radius only. Thus, we will need to select the columns `Symmetry` and `Radius` and `Class`. Save these 5 rows and 3 columns to a data frame named `small_sample`.

Fill in the `___` in the scaffolding provided below.

In [166]:
small_sample = cancer.sample(5, random_state=20)[["Symmetry", "Radius", "Class"]]

small_sample

,Symmetry,Radius,Class
269,-0.478047,-0.947437,Benign
263,-0.351971,0.339484,Malignant
30,0.878077,1.423643,Malignant
452,-0.733432,-0.657776,Benign
214,2.947016,0.122239,Malignant


In [167]:
from hashlib import sha1
assert sha1(str(type(small_sample is None)).encode("utf-8")+b"d302a").hexdigest() == "36105f2046e1487e4b54e2a1bc8a2b3fcb444ba0", "type of small_sample is None is not bool. small_sample is None should be a bool"
assert sha1(str(small_sample is None).encode("utf-8")+b"d302a").hexdigest() == "734e0c4c615e95e0e5830a2cea4ad5b55884ed60", "boolean value of small_sample is None is not correct"

assert sha1(str(type(small_sample)).encode("utf-8")+b"d302b").hexdigest() == "3e6c30ad11a7369a29f1275b53f7e7a81c5fdf3c", "type of type(small_sample) is not correct"

assert sha1(str(type(small_sample.shape)).encode("utf-8")+b"d302c").hexdigest() == "5f663aba53527d6031a1cdf9575c0d43b198f3b4", "type of small_sample.shape is not tuple. small_sample.shape should be a tuple"
assert sha1(str(len(small_sample.shape)).encode("utf-8")+b"d302c").hexdigest() == "8fe51322cedf91523edc7359420871643d75dd1c", "length of small_sample.shape is not correct"
assert sha1(str(sorted(map(str, small_sample.shape))).encode("utf-8")+b"d302c").hexdigest() == "d103c4e19dd1f877e3f722f135c15b940f03ec97", "values of small_sample.shape are not correct"
assert sha1(str(small_sample.shape).encode("utf-8")+b"d302c").hexdigest() == "bff260b96936cbbb6ec86ca3b9f68263575c8f57", "order of elements of small_sample.shape is not correct"

assert sha1(str(type("Symmetry" in small_sample.columns)).encode("utf-8")+b"d302d").hexdigest() == "bf45914311432bcdf3d6b5b7bb548b695cb44165", "type of \"Symmetry\" in small_sample.columns is not bool. \"Symmetry\" in small_sample.columns should be a bool"
assert sha1(str("Symmetry" in small_sample.columns).encode("utf-8")+b"d302d").hexdigest() == "315511158546be50c8dd869e265773ac96263061", "boolean value of \"Symmetry\" in small_sample.columns is not correct"

assert sha1(str(type("Radius" in small_sample.columns)).encode("utf-8")+b"d302e").hexdigest() == "902e68aa8790e58afe05b4e08c7edd36cef70163", "type of \"Radius\" in small_sample.columns is not bool. \"Radius\" in small_sample.columns should be a bool"
assert sha1(str("Radius" in small_sample.columns).encode("utf-8")+b"d302e").hexdigest() == "3164d11f3d282ec79d43ff200276777cbe948e4a", "boolean value of \"Radius\" in small_sample.columns is not correct"

assert sha1(str(type("Class" in small_sample.columns)).encode("utf-8")+b"d302f").hexdigest() == "f9bedb139a8abaf38e63f884aa86de7cb55fb8ce", "type of \"Class\" in small_sample.columns is not bool. \"Class\" in small_sample.columns should be a bool"
assert sha1(str("Class" in small_sample.columns).encode("utf-8")+b"d302f").hexdigest() == "72fca9d447de6ee83d002f725218cd39b25b0fe2", "boolean value of \"Class\" in small_sample.columns is not correct"

print('Success!')

Success!


**Question 2.0.1**
<br> {points: 1}

Finally, create a scatter plot where `Symmetry` is on the x-axis, and `Radius` is on the y-axis. Color the points by `Class`. Name your plot `small_sample_plot`.

Fill in the `___` in the scaffolding provided below.

As you create this plot, ensure you follow the guidelines for creating effective visualizations. In particular, note on the plot axes whether the data is standardized or not. Don't set an opacity value in this chart.

In [168]:
small_sample_plot = alt.Chart(small_sample).mark_point().encode(
    x=alt.X('Symmetry').title('Symmetry'),
    y=alt.Y('Radius').title('Radius'),
    color=alt.Color('Class').title('Diagnosis')
)


small_sample_plot

alt.Chart(...)

In [169]:
from hashlib import sha1
assert sha1(str(type(small_sample_plot is None)).encode("utf-8")+b"ec72").hexdigest() == "6ea091003708fa28d8176ec1ef981f06db70ce31", "type of small_sample_plot is None is not bool. small_sample_plot is None should be a bool"
assert sha1(str(small_sample_plot is None).encode("utf-8")+b"ec72").hexdigest() == "2e5dafc9f3957af6dbe8749a3c954f80b3def855", "boolean value of small_sample_plot is None is not correct"

assert sha1(str(type(sum(small_sample_plot.data.Symmetry))).encode("utf-8")+b"ec73").hexdigest() == "a5ddc2e4e1dd2a7e1eb432b525efb624404fb39c", "type of sum(small_sample_plot.data.Symmetry) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(small_sample_plot.data.Symmetry), 2)).encode("utf-8")+b"ec73").hexdigest() == "59e1204521923517aaa91c747b52e4f147fedcf8", "value of sum(small_sample_plot.data.Symmetry) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(small_sample_plot.mark)).encode("utf-8")+b"ec74").hexdigest() == "a82e9c63fb6197fabc6893813804bb9fbe9c504e", "type of small_sample_plot.mark is not str. small_sample_plot.mark should be an str"
assert sha1(str(len(small_sample_plot.mark)).encode("utf-8")+b"ec74").hexdigest() == "900807d2b4fa4c95bf6cfc2506fbbc84e76aa937", "length of small_sample_plot.mark is not correct"
assert sha1(str(small_sample_plot.mark.lower()).encode("utf-8")+b"ec74").hexdigest() == "5800021557653216e215be8c19fc2a3106b6c6a9", "value of small_sample_plot.mark is not correct"
assert sha1(str(small_sample_plot.mark).encode("utf-8")+b"ec74").hexdigest() == "5800021557653216e215be8c19fc2a3106b6c6a9", "correct string value of small_sample_plot.mark but incorrect case of letters"

assert sha1(str(type(small_sample_plot.encoding.color['shorthand'])).encode("utf-8")+b"ec75").hexdigest() == "9e39fe7ac2103601e8eafa3aee712308a38e7c45", "type of small_sample_plot.encoding.color['shorthand'] is not str. small_sample_plot.encoding.color['shorthand'] should be an str"
assert sha1(str(len(small_sample_plot.encoding.color['shorthand'])).encode("utf-8")+b"ec75").hexdigest() == "5a9f4e4de6ed97f7836ffe86141ad66451d567fb", "length of small_sample_plot.encoding.color['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.color['shorthand'].lower()).encode("utf-8")+b"ec75").hexdigest() == "28851a6baff6d9fa6ebd7175baacb1c3a28cbd28", "value of small_sample_plot.encoding.color['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.color['shorthand']).encode("utf-8")+b"ec75").hexdigest() == "25a3c4d0396b73e13e2b189b0d1b6eba9914aba3", "correct string value of small_sample_plot.encoding.color['shorthand'] but incorrect case of letters"

assert sha1(str(type(small_sample_plot.encoding.x['shorthand'])).encode("utf-8")+b"ec76").hexdigest() == "5b7f94dd0406e2f26eb45dfcdf57741f28d2df95", "type of small_sample_plot.encoding.x['shorthand'] is not str. small_sample_plot.encoding.x['shorthand'] should be an str"
assert sha1(str(len(small_sample_plot.encoding.x['shorthand'])).encode("utf-8")+b"ec76").hexdigest() == "f194bbc46edadb84987695ddb4f1400d288c4f63", "length of small_sample_plot.encoding.x['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.x['shorthand'].lower()).encode("utf-8")+b"ec76").hexdigest() == "59069b2594fb52963b45895a44f828d2156266c5", "value of small_sample_plot.encoding.x['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.x['shorthand']).encode("utf-8")+b"ec76").hexdigest() == "aa4ed72059e3c3822a9420a5330256bc8503fbc4", "correct string value of small_sample_plot.encoding.x['shorthand'] but incorrect case of letters"

assert sha1(str(type(small_sample_plot.encoding.y['shorthand'])).encode("utf-8")+b"ec77").hexdigest() == "8d8b11a250c424e757146ae38c45a53d8a7a9a82", "type of small_sample_plot.encoding.y['shorthand'] is not str. small_sample_plot.encoding.y['shorthand'] should be an str"
assert sha1(str(len(small_sample_plot.encoding.y['shorthand'])).encode("utf-8")+b"ec77").hexdigest() == "afdb03a260f02df6f491aa14833a17c929ffb867", "length of small_sample_plot.encoding.y['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.y['shorthand'].lower()).encode("utf-8")+b"ec77").hexdigest() == "b20d422ef0e84b4abde374cee0454a4fcb1d6c43", "value of small_sample_plot.encoding.y['shorthand'] is not correct"
assert sha1(str(small_sample_plot.encoding.y['shorthand']).encode("utf-8")+b"ec77").hexdigest() == "84689b7e7eb1e4708948acd344618a2386e2bdc8", "correct string value of small_sample_plot.encoding.y['shorthand'] but incorrect case of letters"

assert sha1(str(type(isinstance(small_sample_plot.encoding.x['title'], str))).encode("utf-8")+b"ec78").hexdigest() == "ccea7c9dc50ec9437966a917af97db89d738f7b0", "type of isinstance(small_sample_plot.encoding.x['title'], str) is not bool. isinstance(small_sample_plot.encoding.x['title'], str) should be a bool"
assert sha1(str(isinstance(small_sample_plot.encoding.x['title'], str)).encode("utf-8")+b"ec78").hexdigest() == "8a52586734dd6931c8bbe573feb32b241d302663", "boolean value of isinstance(small_sample_plot.encoding.x['title'], str) is not correct"

assert sha1(str(type(isinstance(small_sample_plot.encoding.y['title'], str))).encode("utf-8")+b"ec79").hexdigest() == "3c124e937ecbdf40eafa778036b0daef2dd0fb43", "type of isinstance(small_sample_plot.encoding.y['title'], str) is not bool. isinstance(small_sample_plot.encoding.y['title'], str) should be a bool"
assert sha1(str(isinstance(small_sample_plot.encoding.y['title'], str)).encode("utf-8")+b"ec79").hexdigest() == "f12963e43ee0e86069778c58602fc008a33bc2b6", "boolean value of isinstance(small_sample_plot.encoding.y['title'], str) is not correct"

assert sha1(str(type(isinstance(small_sample_plot.encoding.color['title'], str))).encode("utf-8")+b"ec7a").hexdigest() == "9c2018be36cd229509a2dfd27032a776b6572065", "type of isinstance(small_sample_plot.encoding.color['title'], str) is not bool. isinstance(small_sample_plot.encoding.color['title'], str) should be a bool"
assert sha1(str(isinstance(small_sample_plot.encoding.color['title'], str)).encode("utf-8")+b"ec7a").hexdigest() == "83933ee5019864f31209d69bd29b1d749ad0f315", "boolean value of isinstance(small_sample_plot.encoding.color['title'], str) is not correct"

print('Success!')

Success!


**Question 2.1** 
<br> {points: 1}

Suppose we are interested in classifying a new observation with `Symmetry = 0.5` and `Radius = 0`, but unknown `Class`. Using the `small_sample` data frame, add another row with `Symmetry = 0.5`, `Radius = 0`, and `Class = "unknown"`.

Fill in the `___` in the scaffolding provided below.

*Assign your answer to an object called `updated_sample`.*

In [170]:
new_observation = pd.DataFrame({"Symmetry": [0.5], "Radius": [0], "Class": ["unknown"]})
updated_sample = pd.concat([small_sample, new_observation]).reset_index(drop=True)

updated_sample

,Symmetry,Radius,Class
0,-0.478047,-0.947437,Benign
1,-0.351971,0.339484,Malignant
2,0.878077,1.423643,Malignant
3,-0.733432,-0.657776,Benign
4,2.947016,0.122239,Malignant
5,0.500000,0.000000,unknown


In [171]:
from hashlib import sha1
assert sha1(str(type(updated_sample is None)).encode("utf-8")+b"63a8f").hexdigest() == "f912e8ad241f18785dbeb02cc5b75c5c45f75d05", "type of updated_sample is None is not bool. updated_sample is None should be a bool"
assert sha1(str(updated_sample is None).encode("utf-8")+b"63a8f").hexdigest() == "4138545a3acad28e9d4701612477afb8e98cb770", "boolean value of updated_sample is None is not correct"

assert sha1(str(type(updated_sample)).encode("utf-8")+b"63a90").hexdigest() == "5d0f898ed1d6b14f33e594fde9fc7bbc7be77977", "type of type(updated_sample) is not correct"

assert sha1(str(type(updated_sample.Class[5])).encode("utf-8")+b"63a91").hexdigest() == "1cd4de1de008519f24c1528c8718df280ae498a8", "type of updated_sample.Class[5] is not str. updated_sample.Class[5] should be an str"
assert sha1(str(len(updated_sample.Class[5])).encode("utf-8")+b"63a91").hexdigest() == "f8554bc540be7cc9c1f79023e31505135c7263cb", "length of updated_sample.Class[5] is not correct"
assert sha1(str(updated_sample.Class[5].lower()).encode("utf-8")+b"63a91").hexdigest() == "8456bb521b4eb5ca91a871546dd257d3582e5fe0", "value of updated_sample.Class[5] is not correct"
assert sha1(str(updated_sample.Class[5]).encode("utf-8")+b"63a91").hexdigest() == "8456bb521b4eb5ca91a871546dd257d3582e5fe0", "correct string value of updated_sample.Class[5] but incorrect case of letters"

assert sha1(str(type(updated_sample.shape)).encode("utf-8")+b"63a92").hexdigest() == "c1e8d67dd7057197b2e451953136fd88fdc7304a", "type of updated_sample.shape is not tuple. updated_sample.shape should be a tuple"
assert sha1(str(len(updated_sample.shape)).encode("utf-8")+b"63a92").hexdigest() == "1c28cd167e28ab145d218e27c6dc1fe68b71bf1b", "length of updated_sample.shape is not correct"
assert sha1(str(sorted(map(str, updated_sample.shape))).encode("utf-8")+b"63a92").hexdigest() == "dc61683c862e7aa5b4f769a8c48decc4d37669bc", "values of updated_sample.shape are not correct"
assert sha1(str(updated_sample.shape).encode("utf-8")+b"63a92").hexdigest() == "5243148f00d7fe409224ec3ad201609b7faaa6c5", "order of elements of updated_sample.shape is not correct"

assert sha1(str(type(sum(updated_sample.Radius))).encode("utf-8")+b"63a93").hexdigest() == "8d0f0c72861df19f0ba69d808a389c3a9f47b699", "type of sum(updated_sample.Radius) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(updated_sample.Radius), 2)).encode("utf-8")+b"63a93").hexdigest() == "b152b066f0c8c565b1959793f13ca08722d8ffc9", "value of sum(updated_sample.Radius) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(updated_sample.Symmetry))).encode("utf-8")+b"63a94").hexdigest() == "b0aa6d4b14f77c07eb7d4f313b7b264720808572", "type of sum(updated_sample.Symmetry) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(updated_sample.Symmetry), 2)).encode("utf-8")+b"63a94").hexdigest() == "15fb68a939aac396d56387529e7068c81b2532c1", "value of sum(updated_sample.Symmetry) is not correct (rounded to 2 decimal places)"

print('Success!')

Success!


**Question 2.2**
<br> {points: 1}

Compute the distance between each pair of the 6 observations in the `updated_sample` dataframe using the `euclidean_distances` function based on two variables: `Symmetry` and `Radius`. Fill in the `___` in the scaffolding provided below.


*Assign your answer to an object called `dist_matrix`.*

In [175]:
dist_matrix = pd.DataFrame(euclidean_distances(updated_sample[["Symmetry", "Radius"]]))

dist_matrix

,0,1,2,3,4,5
0,0.000000,1.293082,2.731500,0.386167,3.588212,1.361695
1,1.293082,0.000000,1.639640,1.067727,3.306133,0.917118
2,2.731500,1.639640,0.000000,2.632350,2.444209,1.472991
3,0.386167,1.067727,2.632350,0.000000,3.762196,1.397864
4,3.588212,3.306133,2.444209,3.762196,0.000000,2.450067
5,1.361695,0.917118,1.472991,1.397864,2.450067,0.000000


In [176]:
from hashlib import sha1
assert sha1(str(type(dist_matrix is None)).encode("utf-8")+b"f1600").hexdigest() == "627d060490109c77bd779cb9e95b0c68036c812b", "type of dist_matrix is None is not bool. dist_matrix is None should be a bool"
assert sha1(str(dist_matrix is None).encode("utf-8")+b"f1600").hexdigest() == "a171a0896e31a3ce91525bdc015d52ea79dbe284", "boolean value of dist_matrix is None is not correct"

assert sha1(str(type(dist_matrix)).encode("utf-8")+b"f1601").hexdigest() == "10968d6906c926d692511390f0232726333366e6", "type of type(dist_matrix) is not correct"

assert sha1(str(type(dist_matrix.shape)).encode("utf-8")+b"f1602").hexdigest() == "1c25efba88f200a7e9193196996808e42fc35b33", "type of dist_matrix.shape is not tuple. dist_matrix.shape should be a tuple"
assert sha1(str(len(dist_matrix.shape)).encode("utf-8")+b"f1602").hexdigest() == "e0d849dd57f0ac1f4e1161dcb710ce15a4aca5ed", "length of dist_matrix.shape is not correct"
assert sha1(str(sorted(map(str, dist_matrix.shape))).encode("utf-8")+b"f1602").hexdigest() == "a607b4975580a68f9535a90a3ff8d75028005422", "values of dist_matrix.shape are not correct"
assert sha1(str(dist_matrix.shape).encode("utf-8")+b"f1602").hexdigest() == "dc39d4872e8dc8c03a8eb6c6e9e83b142aa90122", "order of elements of dist_matrix.shape is not correct"

assert sha1(str(type(sum(dist_matrix.iloc[0]))).encode("utf-8")+b"f1603").hexdigest() == "6e7e2d52109733b70724d905e0a7f809a8b923a6", "type of sum(dist_matrix.iloc[0]) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(dist_matrix.iloc[0]), 2)).encode("utf-8")+b"f1603").hexdigest() == "9f090fe37c54a014b2528edec1949d5eac3a2d78", "value of sum(dist_matrix.iloc[0]) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(dist_matrix.iloc[1]))).encode("utf-8")+b"f1604").hexdigest() == "103f6f3f5c5d0c3e8834c7eab3e6fe434c3398b8", "type of sum(dist_matrix.iloc[1]) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(dist_matrix.iloc[1]), 2)).encode("utf-8")+b"f1604").hexdigest() == "49506abce6dfb47030a6af475a52ebb7eb6aac88", "value of sum(dist_matrix.iloc[1]) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(dist_matrix.iloc[4]))).encode("utf-8")+b"f1605").hexdigest() == "7be096b0b58ce68439ca9331aac4c0890ac351a1", "type of sum(dist_matrix.iloc[4]) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(dist_matrix.iloc[4]), 2)).encode("utf-8")+b"f1605").hexdigest() == "70660e2b811d494704b5b63bc00335a24f5e7e47", "value of sum(dist_matrix.iloc[4]) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(dist_matrix.iloc[5]))).encode("utf-8")+b"f1606").hexdigest() == "e8d7e8545962d1bed827aec9673a59e4cafb3575", "type of sum(dist_matrix.iloc[5]) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(dist_matrix.iloc[5]), 2)).encode("utf-8")+b"f1606").hexdigest() == "31d3450661df48a6063b8a0d0f8aa7ac281bcf45", "value of sum(dist_matrix.iloc[5]) is not correct (rounded to 2 decimal places)"

print('Success!')

Success!


**Question 2.3** Multiple Choice:
<br> {points: 1}

In the table above, the row and column numbers reflect the row number from the data frame the `euclidean_distances` function was applied to. The values in `dist_matrix` are the distances between the points of the row and column number. For example, the distance between the point 2 and point 4 is 2.444209. And the distance between point 3 and point 3 (the same point) is 0.

The diagonal is all zeros since the distance between a point and itself is always zero. Each pairwise distance value occurs two times in the table, once above and once below the diagonal of zeros, so technically it would be enough if we saved the diagonal plus either the values above or below, but here we keep all of them because it fits the table format nicely.

**Which observation is the nearest to our new point?**

*Assign your answer to an object called `answer2_3`. Make sure your answer is a number.

In [177]:
answer2_3 = 1

In [178]:
from hashlib import sha1
assert sha1(str(type(answer2_3)).encode("utf-8")+b"82534").hexdigest() == "ab8ae573f769467e4d9fa25c305ee555e6927b0c", "type of answer2_3 is not int. Please make sure it is int and not np.int64, etc. You can cast your value into an int using int()"
assert sha1(str(answer2_3).encode("utf-8")+b"82534").hexdigest() == "bf25d1aa852ab2185b4b437c852a2ff4d62c7f27", "value of answer2_3 is not correct"

print('Success!')

Success!


**Question 2.4** Multiple Choice: 
<br> {points: 1}

If we use the K-nearest neighbour classification algorithm with K = 1 to classify the new observation using your answers to **Questions 2.2 & 2.3**, is the new data point predicted to be benign or malignant?

*Assign your answer to an object called `answer2_4`. Make sure the correct answer is written fully as either "Benign" or "Malignant". Remember to surround your answer with quotation marks.* 

In [183]:
answer2_4 = "Malignant"

In [184]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X=small_sample[["Symmetry", "Radius"]], y=small_sample["Class"]);
new_obs = pd.DataFrame({"Symmetry": [0.5], "Radius": [0]})
knn.predict(new_obs)

array(['Malignant'], dtype=object)

In [185]:
from hashlib import sha1
assert sha1(str(type(answer2_4)).encode("utf-8")+b"7c755").hexdigest() == "5c51221fe041156e423f38bd3cefadeb88ee0c49", "type of answer2_4 is not str. answer2_4 should be an str"
assert sha1(str(len(answer2_4)).encode("utf-8")+b"7c755").hexdigest() == "d4504fb9530371871bfc36e4abbbdbbddfb13483", "length of answer2_4 is not correct"
assert sha1(str(answer2_4.lower()).encode("utf-8")+b"7c755").hexdigest() == "d145d9d06134b6bace45fed6257b4a5a717e025b", "value of answer2_4 is not correct"
assert sha1(str(answer2_4).encode("utf-8")+b"7c755").hexdigest() == "de3e417b356a359dc7ef8f54be904b3e6b668855", "correct string value of answer2_4 but incorrect case of letters"

print('Success!')

Success!


**Question 2.5** Multiple Choice:
<br> {points: 1}

Using your answers to **Questions 2.2 & 2.3**, what are the three closest observations to your new point?

A. 1, 3, 2

B. 0, 1, 3

C. 5, 2, 4

D. 3, 4, 2

*Assign your answer to an object called `answer2_5`. Make sure the correct answer is an uppercase letter. Remember to surround your answer with quotation marks (e.g. "F").*

In [186]:
answer2_5 = 'B'

In [187]:
from hashlib import sha1
assert sha1(str(type(answer2_5)).encode("utf-8")+b"44a0e").hexdigest() == "c1473050b81d4dc5965d112162398eb89d3a2795", "type of answer2_5 is not str. answer2_5 should be an str"
assert sha1(str(len(answer2_5)).encode("utf-8")+b"44a0e").hexdigest() == "4e478acef365912cb300913c4085c51c315a6936", "length of answer2_5 is not correct"
assert sha1(str(answer2_5.lower()).encode("utf-8")+b"44a0e").hexdigest() == "81c38878a93515dae56ca44549c530253658eb95", "value of answer2_5 is not correct"
assert sha1(str(answer2_5).encode("utf-8")+b"44a0e").hexdigest() == "e39a3081dfb44a8bdd74f14dbb2d0996b5e27f8c", "correct string value of answer2_5 but incorrect case of letters"

print('Success!')

Success!


**Question 2.6** Multiple Choice: 
<br> {points: 1}

We will now use the K-nearest neighbour classification algorithm with K = 3 to classify the new observation using your answers to **Questions 2.2 & 2.3**. Is the new data point predicted to be benign or malignant?

*Assign your answer to an object called `answer2_6`. Make sure the correct answer is written fully. Remember to surround your answer with quotation marks (e.g. "Benign" / "Malignant").*

In [188]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X=small_sample[["Symmetry", "Radius"]], y=small_sample["Class"]);
new_obs = pd.DataFrame({"Symmetry": [0.5], "Radius": [0]})
knn.predict(new_obs)

array(['Benign'], dtype=object)

In [189]:
answer2_6 = "Benign"

In [190]:
from hashlib import sha1
assert sha1(str(type(answer2_6)).encode("utf-8")+b"a609f").hexdigest() == "e3f33cd3175f13c5b1fd35d949dbdee86ab1d2ce", "type of answer2_6 is not str. answer2_6 should be an str"
assert sha1(str(len(answer2_6)).encode("utf-8")+b"a609f").hexdigest() == "5e1dda6237dee7b10ec63a573a4014299b20c18c", "length of answer2_6 is not correct"
assert sha1(str(answer2_6.lower()).encode("utf-8")+b"a609f").hexdigest() == "65018aace771bcd005611919cd31b9b25e0cee53", "value of answer2_6 is not correct"
assert sha1(str(answer2_6).encode("utf-8")+b"a609f").hexdigest() == "3d76f86ce6ddcc905661e3dbb121f5770596e4a5", "correct string value of answer2_6 but incorrect case of letters"

print('Success!')

Success!


**Question 2.7**
<br> {points: 1}

Compare your answers in 2.4 and 2.6. Are they the same?

*Assign your answer to an object called `answer2_7`. Make sure the correct answer is written in lower-case. Remember to surround your answer with quotation marks (e.g. "yes" / "no").* 

In [191]:
answer2_7 = "no"

In [192]:
from hashlib import sha1
assert sha1(str(type(answer2_7)).encode("utf-8")+b"47bd1").hexdigest() == "8089cc982d72a96e540b358961e2843607ea72f9", "type of answer2_7 is not str. answer2_7 should be an str"
assert sha1(str(len(answer2_7)).encode("utf-8")+b"47bd1").hexdigest() == "c832e32ce20e671db031a0bb4d73f75bc54a4d68", "length of answer2_7 is not correct"
assert sha1(str(answer2_7.lower()).encode("utf-8")+b"47bd1").hexdigest() == "4f5f95da1522b173ac00b5a5094b0b530e51a963", "value of answer2_7 is not correct"
assert sha1(str(answer2_7).encode("utf-8")+b"47bd1").hexdigest() == "4f5f95da1522b173ac00b5a5094b0b530e51a963", "correct string value of answer2_7 but incorrect case of letters"

print('Success!')

Success!


## 3. Using `scikit-learn` to perform k-nearest neighbours

Now that we understand how K-nearest neighbours (k-nn) classification works, let's get familar with the `scikit-learn` Python package. The benefit of using `scikit-learn` is that it will keep our code simple, readable and accurate. Coding less and in a tidier format means that there is less chance for errors to occur.  

We'll again focus on `Radius` and `Symmetry` as the two predictors. This time, we would like to predict the class of a new observation with `Symmetry = 1` and `Radius = 0`. This one is a bit tricky to do visually from the plot below, and so is a motivating example for us to compute the prediction using k-nn with the `scikit-learn` package. Let's use `K = 7`.

In [193]:
# Run this to remind yourself what the data looks like
cancer_plot

alt.Chart(...)

**Question 3.1** 
<br> {points: 1}

Create a **model** for K-nearest neighbours classification by using the `KNeighborsClassifier` function. Specify that we want to set `n_neighbors = 7`.

Name your model specification `knn_spec`.

In [217]:
knn_spec = KNeighborsClassifier(n_neighbors=7)


knn_spec


KNeighborsClassifier(n_neighbors=7)

In [218]:
from hashlib import sha1
assert sha1(str(type(knn_spec is None)).encode("utf-8")+b"2245").hexdigest() == "78a85e8a7790e4e7a7d20d518a6e9e23249aeb08", "type of knn_spec is None is not bool. knn_spec is None should be a bool"
assert sha1(str(knn_spec is None).encode("utf-8")+b"2245").hexdigest() == "b44afb32e99c75fdbfb62a424921d1a2fa6f12ee", "boolean value of knn_spec is None is not correct"

assert sha1(str(type(knn_spec.n_neighbors)).encode("utf-8")+b"2246").hexdigest() == "1458700d2adfd82356dfd982b7941979cbd45589", "type of knn_spec.n_neighbors is not int. Please make sure it is int and not np.int64, etc. You can cast your value into an int using int()"
assert sha1(str(knn_spec.n_neighbors).encode("utf-8")+b"2246").hexdigest() == "22e82178737bce10c71283d1cf37544dd1dcbbb8", "value of knn_spec.n_neighbors is not correct"

assert sha1(str(type(knn_spec.algorithm)).encode("utf-8")+b"2247").hexdigest() == "c71db7d462b7abd56ce64492cec1d48ea9040224", "type of knn_spec.algorithm is not str. knn_spec.algorithm should be an str"
assert sha1(str(len(knn_spec.algorithm)).encode("utf-8")+b"2247").hexdigest() == "fc15a33f2e81ff2a1ebe2cdd52ca95582dc1df46", "length of knn_spec.algorithm is not correct"
assert sha1(str(knn_spec.algorithm.lower()).encode("utf-8")+b"2247").hexdigest() == "6fb8474d83ec0dc215e52cdc08b990751b7d94a9", "value of knn_spec.algorithm is not correct"
assert sha1(str(knn_spec.algorithm).encode("utf-8")+b"2247").hexdigest() == "6fb8474d83ec0dc215e52cdc08b990751b7d94a9", "correct string value of knn_spec.algorithm but incorrect case of letters"

print('Success!')

Success!


**Question 3.2**
<br> {points: 1}

To train the model on the breast cancer dataset, pass `knn_spec` and the `cancer` dataset to the `.fit` function. Specify `Class` as your target variable and the `Symmetry` and `Radius` variables as your predictors. Name your fitted model as `knn_fit`.

In [219]:
X = cancer[["Symmetry", "Radius"]]
y = cancer["Class"]
knn_fit = knn_spec.fit(X, y)




knn_fit

KNeighborsClassifier(n_neighbors=7)

In [220]:
from hashlib import sha1
assert sha1(str(type(knn_fit is None)).encode("utf-8")+b"2bff6").hexdigest() == "56a79c088e5ec5a42b96d6ea0d14cd8712fac03e", "type of knn_fit is None is not bool. knn_fit is None should be a bool"
assert sha1(str(knn_fit is None).encode("utf-8")+b"2bff6").hexdigest() == "268e8a79ae1936524d056ca5169d2c71a72e70a5", "boolean value of knn_fit is None is not correct"

assert sha1(str(type(type(knn_fit))).encode("utf-8")+b"2bff7").hexdigest() == "097a69bbecb57e7e121a5edde688e93d2fbf9d17", "type of type(knn_fit) is not correct"
assert sha1(str(type(knn_fit)).encode("utf-8")+b"2bff7").hexdigest() == "40872fc0b9fdb9d1467bfb606dfb31df50cfe4ea", "value of type(knn_fit) is not correct"

assert sha1(str(type(knn_fit.classes_)).encode("utf-8")+b"2bff8").hexdigest() == "cb4bb5ca40cc7954a382bfa33bd1532a3df2e583", "type of knn_fit.classes_ is not correct"
assert sha1(str(knn_fit.classes_).encode("utf-8")+b"2bff8").hexdigest() == "c422ff7ed5336560783f0ada3740882c912a4d97", "value of knn_fit.classes_ is not correct"

assert sha1(str(type(knn_fit.effective_metric_)).encode("utf-8")+b"2bff9").hexdigest() == "aa9d11a76326cf64bec451997ff4ae28ec4e2a9a", "type of knn_fit.effective_metric_ is not str. knn_fit.effective_metric_ should be an str"
assert sha1(str(len(knn_fit.effective_metric_)).encode("utf-8")+b"2bff9").hexdigest() == "6e6efca67ebf0603a534109614346757c0ef146d", "length of knn_fit.effective_metric_ is not correct"
assert sha1(str(knn_fit.effective_metric_.lower()).encode("utf-8")+b"2bff9").hexdigest() == "49892ddeb7ca1f1c4bf772ff4a585dd10249b813", "value of knn_fit.effective_metric_ is not correct"
assert sha1(str(knn_fit.effective_metric_).encode("utf-8")+b"2bff9").hexdigest() == "49892ddeb7ca1f1c4bf772ff4a585dd10249b813", "correct string value of knn_fit.effective_metric_ but incorrect case of letters"

assert sha1(str(type(knn_fit.n_features_in_)).encode("utf-8")+b"2bffa").hexdigest() == "08548a765b78a776a8b132bc9a51a75e1d960554", "type of knn_fit.n_features_in_ is not int. Please make sure it is int and not np.int64, etc. You can cast your value into an int using int()"
assert sha1(str(knn_fit.n_features_in_).encode("utf-8")+b"2bffa").hexdigest() == "114d53a02f71bf2cea7abe3280555955b16647b8", "value of knn_fit.n_features_in_ is not correct"

assert sha1(str(type(X.columns.values)).encode("utf-8")+b"2bffb").hexdigest() == "bd7b6259f8053a315fca30986b168530209f266d", "type of X.columns.values is not correct"
assert sha1(str(X.columns.values).encode("utf-8")+b"2bffb").hexdigest() == "b7dfb165b13d0e67f16b09e1dc18733df42a65a0", "value of X.columns.values is not correct"

assert sha1(str(type(y.name)).encode("utf-8")+b"2bffc").hexdigest() == "e8b9e2753401141ac2885e2394f32cc9cffa348a", "type of y.name is not str. y.name should be an str"
assert sha1(str(len(y.name)).encode("utf-8")+b"2bffc").hexdigest() == "793a9c94797d9b5899e9865188980d72d080ac2d", "length of y.name is not correct"
assert sha1(str(y.name.lower()).encode("utf-8")+b"2bffc").hexdigest() == "bcffa9a693072729c80524aafde25fd6b259f4c3", "value of y.name is not correct"
assert sha1(str(y.name).encode("utf-8")+b"2bffc").hexdigest() == "788a47e28ae9d6cd3cbf4823a5d93017aa6addd9", "correct string value of y.name but incorrect case of letters"

assert sha1(str(type(sum(X.Symmetry))).encode("utf-8")+b"2bffd").hexdigest() == "903f7c4e29e57fe29f567ef056e34f51335aec78", "type of sum(X.Symmetry) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(X.Symmetry), 2)).encode("utf-8")+b"2bffd").hexdigest() == "9397d4f7e4087a12b928268b533b6f46554e7efa", "value of sum(X.Symmetry) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(X.Radius))).encode("utf-8")+b"2bffe").hexdigest() == "d9df2ec2ea3462b25d7d591366163062bdb4021e", "type of sum(X.Radius) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(X.Radius), 2)).encode("utf-8")+b"2bffe").hexdigest() == "5bd565738755b0a3859dbce84dffc4a36fe6656e", "value of sum(X.Radius) is not correct (rounded to 2 decimal places)"

print('Success!')

Success!


**Question 3.3**
<br>{points: 1}

Now we will make our prediction on the `Class` of a new observation with a `Symmetry` of 1 and a `Radius` of 0. First, create a dataframe with these variables and values and call it `new_obs`. Next, use the `.predict` function to obtain our prediction by passing `knn_fit` and `new_obs` to it. Name your predicted class as `class_prediction`.

In [221]:
new_obs = pd.DataFrame([[1, 0]], columns=['Symmetry', 'Radius'])
class_prediction = knn_fit.predict(new_obs)

class_prediction

array(['Malignant'], dtype=object)

In [222]:
from hashlib import sha1
assert sha1(str(type(new_obs is None)).encode("utf-8")+b"cc461").hexdigest() == "ab0357cbb8bfe0c9949b6d69e4c245d2af0635c6", "type of new_obs is None is not bool. new_obs is None should be a bool"
assert sha1(str(new_obs is None).encode("utf-8")+b"cc461").hexdigest() == "a170f5d0292d586ae590d1baca2f13355fb1c178", "boolean value of new_obs is None is not correct"

assert sha1(str(type(new_obs)).encode("utf-8")+b"cc462").hexdigest() == "46dce56999c53d1950618cc8e55690c524892a78", "type of type(new_obs) is not correct"

assert sha1(str(type(new_obs.Symmetry.values)).encode("utf-8")+b"cc463").hexdigest() == "113c0758f0ab20fd6712d816469f64cbc4243a12", "type of new_obs.Symmetry.values is not correct"
assert sha1(str(new_obs.Symmetry.values).encode("utf-8")+b"cc463").hexdigest() == "a41abcf000cf506078cdf4e306ca8121772da0f6", "value of new_obs.Symmetry.values is not correct"

assert sha1(str(type(new_obs.Radius.values)).encode("utf-8")+b"cc464").hexdigest() == "ca2beddfd89352b63fc45c70ca25260e7271c928", "type of new_obs.Radius.values is not correct"
assert sha1(str(new_obs.Radius.values).encode("utf-8")+b"cc464").hexdigest() == "195b0a370675aea18bf01c541c099428f4e248bd", "value of new_obs.Radius.values is not correct"

assert sha1(str(type(class_prediction is None)).encode("utf-8")+b"cc465").hexdigest() == "1ebd9c72926a2430221b10ae35a374b95040d55f", "type of class_prediction is None is not bool. class_prediction is None should be a bool"
assert sha1(str(class_prediction is None).encode("utf-8")+b"cc465").hexdigest() == "883aa082b8853226ec339ff84fd4744558d7fd3b", "boolean value of class_prediction is None is not correct"

assert sha1(str(type(class_prediction)).encode("utf-8")+b"cc466").hexdigest() == "a0d1780e9e6b14b3a86e32b907408ae8253ceec5", "type of class_prediction is not correct"
assert sha1(str(class_prediction).encode("utf-8")+b"cc466").hexdigest() == "2704eb31133cad075ccb97f6c93ca707487c9642", "value of class_prediction is not correct"

print('Success!')

Success!


**Question 3.4**
<br> {points: 1}

Let's perform K-nearest neighbour classification again, but with three predictors. Use the `scikit-learn` package and `K = 7` to classify a new observation where we measure `Symmetry = 1`, `Radius = 0` and `Concavity = 1`. Use the scaffolding from **Questions 3.2** and **3.3** to help you.

- Pass the same `knn_spec` from before to `fit`, but this time specify `Symmetry`, `Radius`, and `Concavity` as the predictors. Save the predictor as `X_2` and the target as `y_2`. Store the output in `knn_fit_2`. 
- Store the new observation values in an object called `new_obs_2`.
- Store the output of `predict` in an object called `class_prediction_2`.

In [223]:
X_2 = cancer[["Symmetry", "Radius", "Concavity"]]
y_2 = cancer["Class"]
knn_fit_2 = knn_spec.fit(X_2, y_2)
new_obs_2 = pd.DataFrame([[1, 0, 1]], columns=['Symmetry', 'Radius', 'Concavity'])
class_prediction_2 = knn_fit_2.predict(new_obs_2)

class_prediction_2

array(['Malignant'], dtype=object)

In [224]:
from hashlib import sha1
assert sha1(str(type(knn_fit_2 is None)).encode("utf-8")+b"1c52c").hexdigest() == "56fcaa2be4db81bcc9472911429b629cba6101f3", "type of knn_fit_2 is None is not bool. knn_fit_2 is None should be a bool"
assert sha1(str(knn_fit_2 is None).encode("utf-8")+b"1c52c").hexdigest() == "99533bc9488848c2608ec11993eb5cfe3033cdf5", "boolean value of knn_fit_2 is None is not correct"

assert sha1(str(type(knn_fit_2.kneighbors)).encode("utf-8")+b"1c52d").hexdigest() == "3f30af90ec8c1a811004f1225ae54fbd83681019", "type of knn_fit_2.kneighbors is not correct"
assert sha1(str(knn_fit_2.kneighbors).encode("utf-8")+b"1c52d").hexdigest() == "55cb84f4cc156f43e6f8f2b5aa8379f34d7d579d", "value of knn_fit_2.kneighbors is not correct"

assert sha1(str(type(knn_fit_2.effective_metric_)).encode("utf-8")+b"1c52e").hexdigest() == "4a071b6a39a1366f1c50cfff6bf5bafe83bd944b", "type of knn_fit_2.effective_metric_ is not str. knn_fit_2.effective_metric_ should be an str"
assert sha1(str(len(knn_fit_2.effective_metric_)).encode("utf-8")+b"1c52e").hexdigest() == "6949133fe0bc9d23f3f2391308a85eeac883a74a", "length of knn_fit_2.effective_metric_ is not correct"
assert sha1(str(knn_fit_2.effective_metric_.lower()).encode("utf-8")+b"1c52e").hexdigest() == "fed6a9fc14d942c80421836e802e4f459699e299", "value of knn_fit_2.effective_metric_ is not correct"
assert sha1(str(knn_fit_2.effective_metric_).encode("utf-8")+b"1c52e").hexdigest() == "fed6a9fc14d942c80421836e802e4f459699e299", "correct string value of knn_fit_2.effective_metric_ but incorrect case of letters"

assert sha1(str(type(type(knn_fit_2))).encode("utf-8")+b"1c52f").hexdigest() == "c0688283eb347b3a716716a2762a46dfa6000cbd", "type of type(knn_fit_2) is not correct"
assert sha1(str(type(knn_fit_2)).encode("utf-8")+b"1c52f").hexdigest() == "0704b8be28fd8da7859dd6fc8e5148535b75399d", "value of type(knn_fit_2) is not correct"

assert sha1(str(type(knn_fit_2.n_features_in_)).encode("utf-8")+b"1c530").hexdigest() == "3ac6a926c4de23bcae7e9a759aca7cd489f4cad2", "type of knn_fit_2.n_features_in_ is not int. Please make sure it is int and not np.int64, etc. You can cast your value into an int using int()"
assert sha1(str(knn_fit_2.n_features_in_).encode("utf-8")+b"1c530").hexdigest() == "a032550f3adad26547a9e2adaae58acd93a5b36e", "value of knn_fit_2.n_features_in_ is not correct"

assert sha1(str(type(X_2.columns.values)).encode("utf-8")+b"1c531").hexdigest() == "a7b153af47464cec438aaac3a5398422f15c739c", "type of X_2.columns.values is not correct"
assert sha1(str(X_2.columns.values).encode("utf-8")+b"1c531").hexdigest() == "c3fd80e7a523addffb5f040e8560caf25c601447", "value of X_2.columns.values is not correct"

assert sha1(str(type(y_2.name)).encode("utf-8")+b"1c532").hexdigest() == "6ecd85411910b1c1e49f8fb6a6c446122031f3a3", "type of y_2.name is not str. y_2.name should be an str"
assert sha1(str(len(y_2.name)).encode("utf-8")+b"1c532").hexdigest() == "ebad5f09e153e6397258881e828d3d859c85154e", "length of y_2.name is not correct"
assert sha1(str(y_2.name.lower()).encode("utf-8")+b"1c532").hexdigest() == "7c29dd51d86bde999e3497b24a52324ffa88d80b", "value of y_2.name is not correct"
assert sha1(str(y_2.name).encode("utf-8")+b"1c532").hexdigest() == "ff10a06265a833d1274bee055d40fe11d6c1de8f", "correct string value of y_2.name but incorrect case of letters"

assert sha1(str(type(y_2.values)).encode("utf-8")+b"1c533").hexdigest() == "6701ed81a4e9d4e3d605bff672f33abc4cab49ab", "type of y_2.values is not correct"
assert sha1(str(y_2.values).encode("utf-8")+b"1c533").hexdigest() == "1eeb2c2596b3d6987df53c982b8921d368bfa856", "value of y_2.values is not correct"

assert sha1(str(type(sum(X_2.Symmetry))).encode("utf-8")+b"1c534").hexdigest() == "401b25657e3ccc6f373b04c3f73b718954f2f87f", "type of sum(X_2.Symmetry) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(X_2.Symmetry), 2)).encode("utf-8")+b"1c534").hexdigest() == "10b6bfc8d4273cf2cda252fff3ead0fcdb4a13cd", "value of sum(X_2.Symmetry) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(X_2.Radius))).encode("utf-8")+b"1c535").hexdigest() == "bc045aaaa3e654827d701f1a154d82ce3bf159de", "type of sum(X_2.Radius) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(X_2.Radius), 2)).encode("utf-8")+b"1c535").hexdigest() == "ba710607201f7fb546dfbff2bbf8125c43eb507a", "value of sum(X_2.Radius) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(sum(X_2.Concavity))).encode("utf-8")+b"1c536").hexdigest() == "bdd5f789273332ff93a37a929e933342daf588ce", "type of sum(X_2.Concavity) is not float. Please make sure it is float and not np.float64, etc. You can cast your value into a float using float()"
assert sha1(str(round(sum(X_2.Concavity), 2)).encode("utf-8")+b"1c536").hexdigest() == "c83ba58970a37bc501a808908479fa9f85081954", "value of sum(X_2.Concavity) is not correct (rounded to 2 decimal places)"

assert sha1(str(type(new_obs_2 is None)).encode("utf-8")+b"1c537").hexdigest() == "0e6db0897dec3a5a0be1727ee2174838fed8392d", "type of new_obs_2 is None is not bool. new_obs_2 is None should be a bool"
assert sha1(str(new_obs_2 is None).encode("utf-8")+b"1c537").hexdigest() == "3bd0c39ffd477eef3d20295014e9603c292a36db", "boolean value of new_obs_2 is None is not correct"

assert sha1(str(type(new_obs_2)).encode("utf-8")+b"1c538").hexdigest() == "dbd7bddf2ba93297c398200c3aba22eee791cc2b", "type of type(new_obs_2) is not correct"

assert sha1(str(type(new_obs_2.Symmetry.values)).encode("utf-8")+b"1c539").hexdigest() == "f025231cf463eb1b53a6eb94ed504ae8eacbd2b3", "type of new_obs_2.Symmetry.values is not correct"
assert sha1(str(new_obs_2.Symmetry.values).encode("utf-8")+b"1c539").hexdigest() == "ee3ccc35dc76e76f3d3f26ce2af31aba1c00475d", "value of new_obs_2.Symmetry.values is not correct"

assert sha1(str(type(new_obs_2.Radius.values)).encode("utf-8")+b"1c53a").hexdigest() == "88a9bfa1595cca275e5f844a75c8fd0e7e660b85", "type of new_obs_2.Radius.values is not correct"
assert sha1(str(new_obs_2.Radius.values).encode("utf-8")+b"1c53a").hexdigest() == "71bc38e24dca943a89fdba0e143c0c93a97d9dbd", "value of new_obs_2.Radius.values is not correct"

assert sha1(str(type(new_obs_2.Concavity.values)).encode("utf-8")+b"1c53b").hexdigest() == "581e0fa5a3a53d9c88d6df38ab3e022f128d41bb", "type of new_obs_2.Concavity.values is not correct"
assert sha1(str(new_obs_2.Concavity.values).encode("utf-8")+b"1c53b").hexdigest() == "6366ed5e9216d5cf0dab850bfd420217af90487c", "value of new_obs_2.Concavity.values is not correct"

assert sha1(str(type(class_prediction_2 is None)).encode("utf-8")+b"1c53c").hexdigest() == "102656310fdb351c06ebfcd34be208a380387416", "type of class_prediction_2 is None is not bool. class_prediction_2 is None should be a bool"
assert sha1(str(class_prediction_2 is None).encode("utf-8")+b"1c53c").hexdigest() == "2a4bb29e14b26cb4fba6321e00f5ea7264414756", "boolean value of class_prediction_2 is None is not correct"

assert sha1(str(type(class_prediction_2)).encode("utf-8")+b"1c53d").hexdigest() == "0d077a418cd3754263e26adf0dd15e657d98c01a", "type of class_prediction_2 is not correct"
assert sha1(str(class_prediction_2).encode("utf-8")+b"1c53d").hexdigest() == "fa0463880a9a71ddc10bfae3b027f9f4a6da0038", "value of class_prediction_2 is not correct"

print('Success!')

Success!


**Question 3.5**
<br>{points: 1}

Finally, we will perform K-nearest neighbour classification again, using the `scikit-learn` package and `K = 7` to classify a new observation where we use **all the predictors** in our data set (we give you the values in the code below). 

But we first have to do one important thing: we need to remove the ID variable from the analysis (it's not a numerical measurement that we should use for classification). Thankfully, `scikit-learn` provides a nice way of combining data preprocessing and training into a single consistent pipeline.

We will first create a preprocessor to remove the `ID` variable using the `drop` preprocessing step. Since we aren't doing any preprocessing to other columns, we will set the `remainder` parameter to `passthrough`. Do so below using the provided scaffolding. Name the preprocessor object `knn_preprocessor`.


In [225]:
knn_preprocessor = make_column_transformer(
    ("drop", ['ID']),
    remainder='passthrough'
)

knn_preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('drop', 'drop', ['ID'])])

In [226]:
from hashlib import sha1
assert sha1(str(type(knn_preprocessor is None)).encode("utf-8")+b"c9190").hexdigest() == "1aa697ad8121d7a033d091dd1dde11fd9a9048c0", "type of knn_preprocessor is None is not bool. knn_preprocessor is None should be a bool"
assert sha1(str(knn_preprocessor is None).encode("utf-8")+b"c9190").hexdigest() == "fa97b74142ab349c61853efd4aed109620ea4419", "boolean value of knn_preprocessor is None is not correct"

assert sha1(str(type(type(knn_preprocessor))).encode("utf-8")+b"c9191").hexdigest() == "1109d30d9e1570f804f20d5d028ef6afce897841", "type of type(knn_preprocessor) is not correct"
assert sha1(str(type(knn_preprocessor)).encode("utf-8")+b"c9191").hexdigest() == "b7f41df01f0ad5de7c336fbe446f33229e39ffba", "value of type(knn_preprocessor) is not correct"

assert sha1(str(type(knn_preprocessor.get_feature_names_out)).encode("utf-8")+b"c9192").hexdigest() == "bb33b66fca24d58fe1cd06f2a2985674ee8f0698", "type of knn_preprocessor.get_feature_names_out is not correct"
assert sha1(str(knn_preprocessor.get_feature_names_out).encode("utf-8")+b"c9192").hexdigest() == "ee35e154ea1785364b8455f97248689b3f960199", "value of knn_preprocessor.get_feature_names_out is not correct"

print('Success!')

Success!


**Question 3.6**
<br> {points: 1}

Create a **pipeline** that includes the new preprocessor (`knn_preprocessor`) and the model specification (`knn_spec`) using the scaffolding below. Name the pipeline object `knn_pipeline`.

In [234]:
knn_pipeline = make_pipeline(knn_preprocessor, knn_spec)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('drop', 'drop', ['ID'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=7))])

In [235]:
from hashlib import sha1
assert sha1(str(type(knn_pipeline is None)).encode("utf-8")+b"570b0").hexdigest() == "5229d62594055e066352668824c5c3908bb29f74", "type of knn_pipeline is None is not bool. knn_pipeline is None should be a bool"
assert sha1(str(knn_pipeline is None).encode("utf-8")+b"570b0").hexdigest() == "1d2994bb8a065a370b5d478a8087d37e3841c353", "boolean value of knn_pipeline is None is not correct"

assert sha1(str(type(type(knn_pipeline))).encode("utf-8")+b"570b1").hexdigest() == "4b6b2b6e8dcd91c384d5e7b3ffaa5df5ed5fac76", "type of type(knn_pipeline) is not correct"
assert sha1(str(type(knn_pipeline)).encode("utf-8")+b"570b1").hexdigest() == "7cff373362fd36faa9d06c119ebecf6820b812f0", "value of type(knn_pipeline) is not correct"

assert sha1(str(type(knn_pipeline.named_steps.kneighborsclassifier.n_neighbors)).encode("utf-8")+b"570b2").hexdigest() == "c0cf487b545aea403a76ddbbedba6f508f1e6b8f", "type of knn_pipeline.named_steps.kneighborsclassifier.n_neighbors is not int. Please make sure it is int and not np.int64, etc. You can cast your value into an int using int()"
assert sha1(str(knn_pipeline.named_steps.kneighborsclassifier.n_neighbors).encode("utf-8")+b"570b2").hexdigest() == "0c675a0ab79e6e0c763779b13443212fca47db27", "value of knn_pipeline.named_steps.kneighborsclassifier.n_neighbors is not correct"

print('Success!')

Success!


**Question 3.7**
{points: 1}

Finally, `fit` the pipeline and predict the class label for the new observation named `new_obs_all`. Name the `fit` object `knn_fit_all`, and the class prediction `class_prediction_all`. Name the new predictor as `X_3` and the new target as `y_3`.

In [254]:
new_obs_all = pd.DataFrame(
    [[None, 0, 0, 0, 0, 0.5, 0, 1, 0, 1, 0]],
    columns=[
        "ID",
        "Radius",
        "Texture",
        "Perimeter",
        "Area",
        "Smoothness",
        "Compactness",
        "Concavity",
        "Concave_points",
        "Symmetry",
        "Fractal_dimension",
    ],
)
X_3 = cancer.drop(columns=['Class'])
y_3 = cancer['Class']
knn_fit_all = knn_pipeline.fit(X_3, y_3)
class_prediction_all = knn_fit_all.predict(new_obs_all)

class_prediction_all

array(['Benign'], dtype=object)

In [255]:
from hashlib import sha1
assert sha1(str(type(class_prediction_all)).encode("utf-8")+b"bdab9").hexdigest() == "6a39e4695fdfb5df240814e4336bf420375f8ab0", "type of class_prediction_all is not correct"
assert sha1(str(class_prediction_all).encode("utf-8")+b"bdab9").hexdigest() == "f3a6c0bfa60af2a0f2398b36ffe3f27359ac2478", "value of class_prediction_all is not correct"

print('Success!')

Success!


## 4. Reviewing Some Concepts

We will conclude with two multiple choice questions to reinforce some key concepts when doing classification with K-nearest neighbours.

**Question 4.0**
<br> {points: 1}

In the K-nearest neighbours classification algorithm, we calculate the distance between the new observation (for which we are trying to predict the class/label/outcome) and each of the observations in the training data set so that we can:

A. Find the `K` nearest neighbours of the new observation

B. Assess how well our model fits the data

C. Find outliers

D. Assign the new observation to a cluster

*Assign your answer (e.g. "E") to an object called: `answer4_0`. Make sure your answer is an uppercase letter and is surrounded with quotation marks.*

In [247]:
answer4_0 = 'A' # finding neighbors 

In [248]:
from hashlib import sha1
assert sha1(str(type(answer4_0)).encode("utf-8")+b"32876").hexdigest() == "2968028cff983f17d6540bc8914f3d47b996a3c6", "type of answer4_0 is not str. answer4_0 should be an str"
assert sha1(str(len(answer4_0)).encode("utf-8")+b"32876").hexdigest() == "96149ef6e9e08b0767c78829d60e48e10c76d82b", "length of answer4_0 is not correct"
assert sha1(str(answer4_0.lower()).encode("utf-8")+b"32876").hexdigest() == "bf7eb488a64fb468f981afb6a22c48e5d9426864", "value of answer4_0 is not correct"
assert sha1(str(answer4_0).encode("utf-8")+b"32876").hexdigest() == "72f0250c431d2044375f10f0e659890a11bb9a27", "correct string value of answer4_0 but incorrect case of letters"

print('Success!')

Success!


**Question 4.1**
<br> {points: 1}

In the K-nearest neighbours classification algorithm, we choose the label/class for a new observation by:

A. Taking the mean (average value) label/class of the K nearest neighbours 

B. Taking the median (middle value) label/class of the K nearest neighbours 

C. Taking the mode (value that appears most often, *i.e.*, the majority vote) label/class of the K nearest neighbours 

*Assign your answer (e.g., "E") to an object called `answer4_1`. Make sure your answer is an uppercase letter and is surrounded with quotation marks.*

In [251]:
answer4_1 = 'C'

In [252]:
from hashlib import sha1
assert sha1(str(type(answer4_1)).encode("utf-8")+b"efb0a").hexdigest() == "fd81d778721d49e0f748a99c8b6630137990c6e3", "type of answer4_1 is not str. answer4_1 should be an str"
assert sha1(str(len(answer4_1)).encode("utf-8")+b"efb0a").hexdigest() == "70fd0074b7e2d3374bdea9e7921a74f40a6076af", "length of answer4_1 is not correct"
assert sha1(str(answer4_1.lower()).encode("utf-8")+b"efb0a").hexdigest() == "1d1126f4f8ef88cb64b68937378a3523069d2dd2", "value of answer4_1 is not correct"
assert sha1(str(answer4_1).encode("utf-8")+b"efb0a").hexdigest() == "4f66d8df3ceed6d93679130380ed8da04b6f60d1", "correct string value of answer4_1 but incorrect case of letters"

print('Success!')

Success!
